# Bronze to Silver Transformation

**Pipeline Stage**: Data Quality & Enrichment  
**Source**: Bronze Layer - 'API_data'

**Target**: Silver Layer - Cleaned and validated API_data

---

## 1. Data Exploration
Initial exploration of schema and data quality.

In [0]:
api_df = spark.sql("SELECT * FROM delta.`/Volumes/telecom_catalog/default/bronze/API_data`")

In [0]:
api_df.display()

In [0]:
api_df.printSchema()

root
 |-- timestamp: timestamp (nullable = true)
 |-- device_id: string (nullable = true)
 |-- active_sessions: long (nullable = true)
 |-- anomaly_score: double (nullable = true)
 |-- cache_hit_ratio: double (nullable = true)
 |-- crash_count: long (nullable = true)
 |-- disk_read_ops: long (nullable = true)
 |-- disk_write_ops: long (nullable = true)
 |-- failed_requests: long (nullable = true)
 |-- fan_speed_rpm: long (nullable = true)
 |-- io_wait_time_ms: long (nullable = true)
 |-- latency_ms_p99: long (nullable = true)
 |-- packet_loss_percentage: double (nullable = true)
 |-- power_usage_watts: long (nullable = true)
 |-- process_count: long (nullable = true)
 |-- queue_length: long (nullable = true)
 |-- reboot_count: long (nullable = true)
 |-- request_count: long (nullable = true)
 |-- service_restart_count: long (nullable = true)
 |-- tcp_connections: long (nullable = true)
 |-- temperature_celsius: double (nullable = true)
 |-- thread_count: long (nullable = true)
 |-- una

## 2. Data Quality Checks
Identify duplicates, null values, and data anomalies.

In [0]:
from pyspark.sql.functions import *

# Total rows
print(api_df.count())

# Duplicate rows
print(api_df.count() - api_df.dropDuplicates().count())

# Duplicate business keys
api_df.groupBy("device_id", "timestamp") \
      .count() \
      .filter("count > 1") \
      .show()

# Null count
display(
    api_df.select([
        count(when(col(c).isNull(), c)).alias(c)
        for c in api_df.columns
    ])
)

1000000
0
+---------+---------+-----+
|device_id|timestamp|count|
+---------+---------+-----+
+---------+---------+-----+



timestamp,device_id,active_sessions,anomaly_score,cache_hit_ratio,crash_count,disk_read_ops,disk_write_ops,failed_requests,fan_speed_rpm,io_wait_time_ms,latency_ms_p99,packet_loss_percentage,power_usage_watts,process_count,queue_length,reboot_count,request_count,service_restart_count,tcp_connections,temperature_celsius,thread_count,unauthorized_access_attempts,uptime_percentage
0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0


## 3. Data Cleaning & Standardization
Standardize business keys and add validation flags.

In [0]:

from pyspark.sql.functions import upper, trim, col

silver_df = (
    api_df
    .withColumn("device_id", upper(trim(col("device_id"))))
)

# Even if today's data is clean, this prevents future inconsistencies.

In [0]:
# Create a validation flag instead of dropping rows:

from pyspark.sql.functions import when, lit

silver_df = silver_df.withColumn(
    "is_valid",
    when(
        (col("anomaly_score").between(0,1)) &
        (col("cache_hit_ratio").between(0,1)) &
        (col("uptime_percentage").between(0,100)) &
        (col("temperature_celsius").between(-20,120)),
        lit(True)
    ).otherwise(lit(False))
)

# This lets you quarantine invalid rows later if needed.

## 4. Feature Engineering
Create derived business metrics for downstream analytics.

In [0]:

silver_df = silver_df.withColumn(
    "temperature_status",
    when(col("temperature_celsius") >= 70, "High")
    .when(col("temperature_celsius") <= 10, "Low")
    .otherwise("Normal")
)

In [0]:

silver_df = silver_df.withColumn(
    "network_health",
    when(
        (col("latency_ms_p99") > 100) |
        (col("packet_loss_percentage") > 2),
        "Poor"
    ).otherwise("Healthy")
)

In [0]:

silver_df = silver_df.withColumn(
    "uptime_category",
    when(col("uptime_percentage") >= 99.9, "Excellent")
    .when(col("uptime_percentage") >= 99, "Good")
    .otherwise("Poor")
)

## 5. Audit Columns
Add metadata for data lineage and troubleshooting.

In [0]:

from pyspark.sql.functions import current_timestamp, to_date

silver_df = (
    silver_df
    .withColumn("silver_load_time", current_timestamp())
    .withColumn("event_date", to_date("timestamp"))
)

## 6. Write to Silver Layer
Persist the cleaned and enriched data to the silver layer.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS telecom_catalog.silver_schema
--MANAGED LOCATION 'pastesilver container url'

In [0]:
# Write to silver layer Delta table
(
    silver_df.write
    .format("delta")
    .mode("append")          # append for incremental loads
    .option("mergeSchema", "true")
    .partitionBy("event_date")
    .save("/Volumes/telecom_catalog/default/silver/API_data")
)

---

## Pipeline Notes

### Key Features
* **Data Quality Flagging**: Invalid records are flagged with `is_valid=False` rather than dropped, enabling downstream analysis
* **Business Logic**: Derived columns classify device status (temperature, network health, uptime) for operational dashboards
* **Audit Trail**: `silver_load_time` and `event_date` support data lineage and partition optimization

### Schema Additions
| Column | Type | Description |
|--------|------|-------------|
| `is_valid` | boolean | Data quality flag (True if all metrics are within acceptable ranges) |
| `temperature_status` | string | High (≥70°C), Low (≤10°C), or Normal |
| `network_health` | string | Poor (latency >100ms OR packet loss >2%) or Healthy |
| `uptime_category` | string | Excellent (≥99.9%), Good (≥99%), or Poor |
| `silver_load_time` | timestamp | ETL processing timestamp |
| `event_date` | date | Event date for partitioning |
